<a href="https://colab.research.google.com/github/kamat-v/qc-course-materials/blob/main/notebooks/week03_bb84_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#A simple implementation of **BB84**

In [1]:
!pip install qiskit qiskit_aer qiskit-ibm-runtime pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.6/378.6 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.4 MB/s eta 0:00:00
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=021df916b795646ab631c61814c0621e05755219fb829b854a92b431f8481ac7
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a17

In [17]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
import numpy as np
import matplotlib.pyplot as plt
import random

**1. Initializing a simulator backend**

In [18]:
random.seed(42)
backend=AerSimulator(seed_simulator=42)
sampler=Sampler(backend)


**2. Preparing a list of bits for Alice to send to Bob.**

In [33]:
# Number of qubits to send
n_qubits = 40

# Alice's random bits and basis choices
alice_bits = [random.choice([0, 1]) for _ in range(n_qubits)]

# Storage for measurements and basis choices
bob_bits = []
alice_used_h = []
bob_used_h = []
eve_used_h = []
eve_bits = []

print("=" * 60)
print("BB84 PROTOCOL WITH EVE'S INTERCEPT-RESEND ATTACK")
print("=" * 60)
print(f"\nSending the following {n_qubits} qubits\n")
print(f"{alice_bits}")



BB84 PROTOCOL WITH EVE'S INTERCEPT-RESEND ATTACK

Sending the following 40 qubits

[1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1]


In [34]:
for bit in alice_bits:
    # ALICE PREPARES
    circuit = QuantumCircuit(1)
    # Alice encodes her bit
    if bit:
        circuit.x(0)
    # Alice randomly chooses a basis (H or Z)
    alice_h = random.choice([True, False])
    if alice_h:
        circuit.h(0)
    circuit.barrier(label="Alice to Eve")

    # EVE INTERCEPTS
    # Eve randomly chooses a measurement basis
    eve_h = random.choice([True, False])
    if eve_h:
        circuit.h(0)

    # Eve measures the qubit
    circuit.measure_all()
    job = sampler.run([circuit], shots=1)
    eve_bit = int(job.result()[0].data.meas.get_bitstrings()[0])

    # EVE RESENDS
    # Eve prepares a new qubit based on her measurement
    circuit = QuantumCircuit(1)
    if eve_bit:
        circuit.x(0)
    if eve_h:
        circuit.h(0)

    circuit.barrier(label="Eve to Bob")

    # BOB MEASURES
    # Bob randomly chooses a measurement basis
    bob_h = random.choice([True, False])
    if bob_h:
        circuit.h(0)

    circuit.measure_all()
    job = sampler.run([circuit], shots=1)
    bob_bit = int(job.result()[0].data.meas.get_bitstrings()[0])

    # Store all results
    bob_bits.append(bob_bit)
    alice_used_h.append(alice_h)
    bob_used_h.append(bob_h)
    eve_used_h.append(eve_h)
    eve_bits.append(eve_bit)



In [35]:
# STEP 1: BASIS RECONCILIATION
sifted_key_alice = []
sifted_key_bob = []
matching_indices = []

for i in range(len(alice_bits)):
    if alice_used_h[i] == bob_used_h[i]:
        sifted_key_alice.append(alice_bits[i])
        sifted_key_bob.append(bob_bits[i])
        matching_indices.append(i)

print(f"Matching bases: {len(sifted_key_alice)}/{n_qubits} qubits")
print(f"Sifted key length: {len(sifted_key_alice)} bits\n")



Matching bases: 16/40 qubits
Sifted key length: 16 bits



In [36]:
# STEP 2: ERROR DETECTION

# Sample about 50% of sifted bits for error checking
sample_size = len(sifted_key_alice) // 2
sample_indices = random.sample(range(len(sifted_key_alice)), sample_size)

errors = 0
for i in sample_indices:
    if sifted_key_alice[i] != sifted_key_bob[i]:
        errors += 1

qber = (errors / sample_size * 100) if sample_size > 0 else 0

print(f"Sampled {sample_size} bits for error checking")
print(f"Errors detected: {errors}/{sample_size}")
print(f"Quantum Bit Error Rate (QBER): {qber:.1f}%\n")

Sampled 8 bits for error checking
Errors detected: 4/8
Quantum Bit Error Rate (QBER): 50.0%



In [37]:
# === STEP 3: SECURITY ANALYSIS ===

if qber > 11:  # Typical threshold for BB84
    print("KEY DISCARDED DUE TO HIGH ERROR RATE!")
else:
    print("KEY ACCEPTED!")

KEY DISCARDED DUE TO HIGH ERROR RATE!


In [38]:
# === FINAL KEY ===
remaining_key_alice = [sifted_key_alice[i] for i in range(len(sifted_key_alice))
                       if i not in sample_indices]
remaining_key_bob = [sifted_key_bob[i] for i in range(len(sifted_key_bob))
                     if i not in sample_indices]

if qber <= 11:
    print(f"Final shared key: {remaining_key_alice}")
    print(f"Final shared key length: {len(remaining_key_alice)} bits")
else:
    print("Final shared key: NONE (protocol aborted due to eavesdropping)")

Final shared key: NONE (protocol aborted due to eavesdropping)
